In [ ]:
%pip install openai scikit-learn

In [ ]:
# Create client

from openai import OpenAI
from sklearn.model_selection import train_test_split

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1", # Ollama
    api_key="sk-1234"  # Dummy key
)

In [ ]:
# Set model parameters

SYSTEM_PROMPT = """
You are for a GAD-7 survey chatbot.
Your job is to extract answers for GAD-7 based on a given transcript.
The Generalized Anxiety Disorder 7-item scale (GAD-7) is a widely used self-administered diagnostic tool designed to screen for and assess the severity of generalized anxiety disorder (GAD).
GAD-7 question order:
q1: Over the last two weeks, how often have you been bothered by feeling nervous, anxious, or on edge?
q2: Over the last two weeks, how often have you been bothered by not being able to stop or control worrying?
q3: Over the last two weeks, how often have you been bothered by worrying too much about different things?
q4: Over the last two weeks, how often have you been bothered by trouble relaxing?
q5: Over the last two weeks, how often have you been bothered by being so restless that it is hard to sit still?
q6: Over the last two weeks, how often have you been bothered by becoming easily annoyed or irritable?
q7: Over the last two weeks, how often have you been bothered by feeling afraid, as if something awful might happen?
Scale mapping:
0 = not at all
1 = several days
2 = more than half the days
3 = nearly every day


Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The final normalized answer for each question must be a single string value: "0", "1", "2", or "3".
Rules:
- For each question q1-q7, determine the score 0-3 based on the user's responses.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q7 key.
- expected_output must contain exactly q1 through q7.
- conversation must be consistent with expected_output.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0"
}
"""
MODEL = 'qwen3.5:2b'

- Use a json dataset of conversation transcript
- Extract scores using LLM or NLP technique
- Validate whether they were correct

In [ ]:
# Load the GAD dataset
import json 

with open('gad_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')

In [ ]:
# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(len(conv_train), len(_conv_test), len(score_train), len(_score_test))

In [ ]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

def create_prediction(transcript: str) -> str:
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            Message(SYSTEM_PROMPT, 'system'),
            Message(transcript)
        ],
    )

    return completion.choices[0].message.content

In [ ]:
outputs = [create_prediction(c) for c in conv_train]

In [ ]:
json_outputs = [json.loads(o) for o in outputs]